# Data preprocessing notebook

## Preprocessing plan — Dataset A (Baron, GSE84133) for one-vs-rest linear SVM

**Status of `data/interim/baron_human.h5ad`: not ready for training as-is.** The data is clean but raw; no preprocessing is applied anywhere in the repo yet.

### Current state (verified)
- 8,569 cells × 20,125 genes, sparse CSR float32, ~90.6% zeros. No NaN/inf/negatives, no duplicate cells or gene names.
- `X` = raw integer UMI counts (identical to `layers['counts']`, see `scripts/build_h5ad.py` L44–48).
- `layers['logcounts']` = curators' scater log2 values computed on **all cells** → **do not use** (leaks across any split).
- Cells were already filtered by the authors (min 1,201 UMIs, 767 detected genes); no doublet tail (`data_preview.ipynb` cell 20).
- No `MT-` genes → no %mito filter possible. `ERCC1…` are real genes, not spike-ins.
- Labels: `obs['cell_type1']`, 14 classes, no unknown labels. Donor: `obs['human']` (1–4).

### Why it is not ready
1. **Raw counts** — an SVM on `X` is driven by library size and a few abundant hormone genes (INS, GCG, SST, REG1A).
2. **Tiny classes** — t_cell 7, schwann 13, epsilon 18 cells (1–5 per donor); largest/smallest class ratio is 361×.
3. **Donor ↔ label confounding** — 843 / 958 acinar cells come from donor 3; non-donor-aware splits are optimistic.
4. **Gene-name mismatch with B/C** — R `make.names` turned `-` into `.` in 513 names (`HLA.A`, `HLA.DRA`, …). The A∩B∩C overlap reported in `data_preview_dataset_B_and_C.ipynb` (15,400) silently drops 116 genes incl. the whole HLA family; after fixing names it is **15,516**.
5. **Leakage-prone artifacts** — `layers['logcounts']` and the throw-away normalize → HVG → scale → PCA in `data_preview.ipynb` cell 29 were fit on all cells; do not reuse.

---

### Part 1 — Stateless, per-cell steps (safe before splitting)
**Where:** new `scripts/preprocess.py` (called from this notebook). Output: `data/processed/baron_human.h5ad` + a JSON of parameters. Keep `scripts/build_h5ad.py` as a lossless converter.

| # | Step | How |
|---|---|---|
| 1 | Fix gene names | `var_names.str.replace('.', '-', regex=False)`; keep originals in `var['symbol_orig']`; assert still unique |
| 2 | Cell QC | ≥ 500 detected genes and ≥ 1,000 UMIs (removes 0 Baron cells, but applies the same rule to B/C). No %MT, no genes-per-UMI filter (would drop acinar), no doublet step |
| 3 | Labels & metadata | `obs['label']` ← `cell_type1`, `obs['donor']` ← `human`, `obs['library']` ← cell-ID prefix (`humanN_libM`), `obs['lineage']` ← map from `data_preview.ipynb` cell 18; drop constant QC columns. Move t_cell, schwann, epsilon to `obs['split'] = 'rare_unseen'` (held-out unseen pool for open-set evaluation); consider flagging mast (25 cells, 1 in donor 4) |
| 4 | Gene set | Restrict to the 15,516 name-fixed A∩B∩C genes (decided by gene names only, not query expression) |
| 5 | Normalize | `sc.pp.normalize_total(target_sum=1e4)` → `sc.pp.log1p`, **after** step 4; keep raw counts in `layers['counts']` |

### Part 2 — Fitted steps (training fold only)
**Where:** inside an sklearn `Pipeline` in the training code (`machine_learning_implementation_lab.ipynb` now, later `scripts/train_svm.py`).

| # | Step | How |
|---|---|---|
| 6 | Min-detection gene filter | keep genes detected in ≥ 10 training cells (small custom transformer) |
| 7 | HVG selection | top 2,000 HVGs on training cells only, batch-aware across training donors; custom `TransformerMixin` storing gene indices |
| 8 | Scaling (optional, test it) | `StandardScaler` on HVGs, clip to ±10 (or `with_mean=False` to stay sparse). No PCA for the SVM (PCA is for the k-NN baseline) |
| 9 | Classifier | `LinearSVC(class_weight='balanced', max_iter>=5000, random_state=SEED)`, C tuned by inner donor-grouped CV. LinearSVC is already one-vs-rest. For rejection probabilities wrap in `CalibratedClassifierCV` with grouped CV |

### Splits and evaluation
- Use `StratifiedGroupKFold(groups=donor)` or leave-one-donor-out; report per-donor class counts per fold.
- Plain stratified 5-fold = optimistic ceiling only.
- Metric: macro-F1.
- A → B/C transfer: fit the whole pipeline on all of A and apply it frozen; **never refit HVG/scaler on the query**.

### Open decisions
- [ ] Exclude t_cell / schwann / epsilon from training (recommended) — open question 2 in the preview notebook.
- [ ] Correct the 15,400 A∩B∩C figure in `data_preview_dataset_B_and_C.ipynb` to 15,516 after the gene-name fix.
